# 🇫🇷 🥐 Discovering Paris with Semantic Search - A Technical Guide to FastEmbed & Qdrant 

## What is Vector Search?

Traditional keyword search works by matching exact words. But this fails when:

- Searching through images, audio, video, or code
- The same concept is phrased in multiple ways
- No explicit keywords exist

**Vector search** converts data into numerical embeddings and finds items with similar meaning using distance metrics (e.g., cosine similarity).

## Why Qdrant?

Qdrant is an **open-source vector search engine** built in Rust. It's designed for production-scale vector search with features like:
- Dense, sparse, and multi-vector support
- Payload filtering and indexing
- Built-in Web UI for visualization

## Why FastEmbed?
FastEmbed is a **CPU-optimized embedding library** that uses ONNX Runtime for fast inference without heavy frameworks like PyTorch. It integrates seamlessly with Qdrant.

# Part 1: Setup, Installation, and Data Loading

## Step 1: Setup Qdrant with Docker

If you haven't already, run Qdrant locally:

```bash
docker run -d -p 6333:6333 -p 6334:6334 \
  -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
  qdrant/qdrant:latest
```
* Port **6333** : HTTP API (for REST calls)
* Port **6334** : gRPC API (for higher performance)
* Web UI: http://localhost:6333/dashboard

## Step 2: Install Required Libraries

In [1]:
! pip install -q "qdrant-client[fastembed]" pandas requests


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Step 3: Import Libraries & Connect to Qdrant

In [2]:
import requests
import pandas as pd
import csv
import ast
    
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding
from requests.exceptions import RequestException
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


/Volumes/TantiK/Blogs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Connect to local Qdrant instance\n",
client = QdrantClient(host="localhost", port=6333)
    
# Verify connection
response = requests.get("http://localhost:6333")
print(response.json())

{'title': 'qdrant - vector search engine', 'version': '1.17.1', 'commit': 'eabee371fda447974a94d29fbaa675a6a596cc7b'}


To perform a simple connectivity check, simply display the list of collections.

In [4]:
client = QdrantClient(host="localhost", port=6333)
print(client.get_collections())

collections=[CollectionDescription(name='french_tourism')]


# Part 2: Data Loading

## Step 4: Study the Dataset

We're using the **Paris Enriched Tourism Dataset** (477 Points of Interest). Each record contains:
- `name` (str) – POI name
- `address` (str) – Street address
- `clean_reviews` (str) – Combined tourist reviews
- `subCategory` (str) – Type (Hotel, Wine Bar, Pub, etc.)
- `lat`, `lng` (float) – Coordinates
- `polarities` (list[int]) – Sentiment scores (0–10)

## Step 5: Choose Fields for Vector Search and Metadata

### Choosing Fields for Vector Search

We concatenate `name`, `address`, and `clean_reviews` because:
- **Name** provides the POI identity
- **Address** adds location context (helps distinguish similar venues)
- **Clean_reviews** contains the semantic content users will query


### Choosing Fields as Payload (Metadata)
We store `subCategory`, `name`, `address`, `lat`, `lng`, and `polarities` as payload. This allows:
- Filtering by category (e.g., "only Wine Bars")
- Geographic constraints (e.g., "near Louvre")
- Sentiment-based ranking (if we aggregate polarity)

## Step 6: Load the Dataset

We will laod a copy of the Paris Enriched Tourism Dataset (POIs). The original version can be accessed at [Mendeley Data](https://data.mendeley.com/datasets/vh4g4g2322/1).

In [5]:
url = 'https://github.com/tantikristanti/Datasets/releases/download/paris-tourism/Paris.csv'

try:
    # Load the dataset
    df = pd.read_csv(url,
                engine="python", # The Python engine is more tolerant of messy CSVs
                on_bad_lines="skip", # Skip corrupted rows instead of crashing
                quoting=csv.QUOTE_MINIMAL, # Force proper quoting handling
                sep=";")
    print(f"Loaded {len(df)} records")
    print(df.head())
except RequestException as e:
    print(f"Error downloading file: {e}")

Loaded 477 records
      id    category  subCategory  \
0  83256  attraction    Hotel Bar   
1  83321  attraction        Hotel   
2  83358  attraction     Wine Bar   
3  83363  attraction  Music Venue   
4  83364  attraction          Pub   

                                               name location  \
0                                       Hôtel Amour    Paris   
1                                     Pershing Hall    Paris   
2                                    Le Baron Rouge    Paris   
3                                           Glazart    Paris   
4  O'Sullivan's by the mill : Backstage by the mill    Paris   

                                  address        lat       lng  \
0                        8 rue de Navarin  48.879610  2.339383   
1                   49 Rue Pierre Charron  48.869146  2.302191   
2                 1 rue Théophile Roussel  48.849623  2.377437   
3  7-15 Avenue de la Porte de la Villette  48.899198  2.386726   
4                  92 Boulevard de Clichy  

In [6]:
df.keys()

Index(['id', 'category', 'subCategory', 'name', 'location', 'address', 'lat',
       'lng', 'reviews', 'keywords', 'keywords_all', 'NOUN', 'NOUN_VERB',
       'NOUN_ADJ', 'NOUN_VERB_ADJ'],
      dtype='str')

In [7]:
df.dtypes

id                 int64
category             str
subCategory          str
name                 str
location             str
address              str
lat              float64
lng              float64
reviews              str
keywords             str
keywords_all         str
NOUN                 str
NOUN_VERB            str
NOUN_ADJ             str
NOUN_VERB_ADJ        str
dtype: object

## Step 7: Extract Review Texts and Metadata

### Extract the Review Text and Metadata

We will extract reviews from tourists along with associated ratings and polarities.

In [8]:
# --- Function to extract review text + metadata ---
def extract_reviews(data):
    if pd.isna(data):
        return "", [], []
  
    try:
        parsed = ast.literal_eval(data)  # convert string → Python object
      
        if isinstance(parsed, list):
            texts = []
            ratings = []
            polarities = []
          
            for item in parsed:
                if isinstance(item, dict):
                    # Extract fields safely
                    text = item.get("text", "")
                    rating = item.get("rating", None)
                    polarity = item.get("polarity", None)
                  
                    if text:
                        texts.append(text)
                    if rating is not None:
                        ratings.append(rating)
                    if polarity is not None:
                        polarities.append(polarity)
          
            # Combine all review texts into one string
            combined_text = " ".join(texts)
          
            return combined_text, ratings, polarities
  
    except Exception:
        pass

    # fallback if parsing fails
    return str(data), [], []


The `ratings` column only contains the value 0, so we won't use it. Instead, we'll use the `polarities` column.

In [9]:
# Apply extraction
df[["clean_reviews", "ratings", "polarities"]] = df["reviews"].apply(
    lambda x: pd.Series(extract_reviews(x))
)

# Optional: Add a scalar rating for filtering
df['avg_polarity'] = df['polarities'].apply(lambda x: sum(x)/len(x) if x else 0)

df[["clean_reviews", "ratings", "polarities", "avg_polarity"]].head()

# Save to file (optional)
#df.to_csv("data/clean_reviews.csv", index=False)

,clean_reviews,ratings,polarities,avg_polarity
0,don't forget to book an after diner room at th...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 8, 5, 10, 5, 5, 10, 10, 5, 10, 10, 5, 1...",7.181818
1,New Bellini Cocktail > good ! Amazing design i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[10, 10, 10, 5, 10, 10, 8, 10, 5, 10, 5, 5, 5,...",8.529412
2,stop by after Marche Aligre to try wine by the...,"[0, 0, 0, 0, 0, 0, 0, 0, 0]","[5, 10, 0, 5, 5, 5, 5, 5, 8]",5.333333
3,"Visit of the new exhibition Maze after ""Labyri...","[0, 0, 0, 0]","[5, 10, 5, 0]",5.000000
4,"Irish can skip the que. Just say 'Kiss me, I'm...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[5, 5, 10, 5, 10, 8, 5, 5, 10, 5, 10, 10, 5, 10]",7.357143


# Part 2: Embedding, Indexing, and Basic Search (Core pipeline)

## Step 8: Choose Embedding Model

Our model selection was guided by several criteria. We need a model that:
- Works with **English** text (our dataset)
- Is **unimodal** (no image support needed; better performance for text-only)
- Produces **small-to-moderate** dimensions (384–512) to fit in memory
- Is **CPU-optimized** (local inference)

**Selected: `BAAI/bge-small-en-v1.5`**
- Dimension: 384
- Size: ~67 MB
- Optimized for cosine similarity
- Supports query/document prefixes (not strictly required but can improve retrieval)
> **Note:** The distance metric must match the one used during model training. For BGE models, `COSINE` is used.

In [10]:
# Verify available models
print(TextEmbedding.list_supported_models())

[{'model': 'BAAI/bge-base-en', 'sources': {'hf': 'Qdrant/fast-bge-base-en', 'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz', '_deprecated_tar_struct': True}, 'model_file': 'model_optimized.onnx', 'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.', 'license': 'mit', 'size_in_GB': 0.42, 'additional_files': [], 'dim': 768, 'tasks': {}}, {'model': 'BAAI/bge-base-en-v1.5', 'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q', 'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz', '_deprecated_tar_struct': True}, 'model_file': 'model_optimized.onnx', 'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.', 'license': 'mit', 'size_in_GB': 0.21, 'additional_files': [], 'dim': 768, 'tasks': {}}, {'model': 'BAAI/bge-large-en-v1.5', 'sources': {'hf': '

## Step 9: Create a Qdrant Collection

A collection is a container for points with a shared vector configuration.

**Parameters:**
- `size`: Must match embedding dimension (384)
- `distance`: `COSINE` (ranges from -1 to 1; higher = more similar)

In [11]:
collection_name = "french_tourism"

# Check if collection exists
collections = client.get_collections().collections
collection_names = [c.name for c in collections]

if collection_name not in collection_names:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created ✅")
else:
    print(f"Collection '{collection_name}' already exists 👍")

Collection 'french_tourism' already exists 👍


## Step 10: Generate Embeddings & Insert Points

Points are the core data entities in Qdrant. Each point in Qdrant has three components:
1. `id` – Unique identifier (int or UUID)
2. `vector` – The embedding (list of floats)
3. `payload` – Optional metadata for filtering

**Batch Upsert:** For production, use `upload_points()` for large datasets. Here we use `upsert()` for simplicity.



In [12]:
df.head()

,id,category,subCategory,name,location,address,lat,lng,reviews,keywords,keywords_all,NOUN,NOUN_VERB,NOUN_ADJ,NOUN_VERB_ADJ,clean_reviews,ratings,polarities,avg_polarity
0,83256,attraction,Hotel Bar,Hôtel Amour,Paris,8 rue de Navarin,48.879610,2.339383,"[{'language': 'en', 'polarity': 5, 'rating': 0...","[[('forget', 'VB'), ('book', 'NN'), ('diner', ...","[['forget', 'book', 'diner', 'room', 'hotel'],...","[['book', 'diner', 'room', 'hotel'], ['place',...","[['forget', 'book', 'diner', 'room', 'hotel'],...","[['book', 'diner', 'room', 'hotel'], ['place',...","[['forget', 'book', 'diner', 'room', 'hotel'],...",don't forget to book an after diner room at th...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 8, 5, 10, 5, 5, 10, 10, 5, 10, 10, 5, 1...",7.181818
1,83321,attraction,Hotel,Pershing Hall,Paris,49 Rue Pierre Charron,48.869146,2.302191,"[{'language': 'en', 'polarity': 10, 'rating': ...","[[('New', 'NNP'), ('Bellini', 'NNP'), ('Cockta...","[['New', 'Bellini', 'Cocktail', '>', 'good'], ...","[['New', 'Bellini', 'Cocktail', '>'], ['design...","[['New', 'Bellini', 'Cocktail', '>'], ['Amazin...","[['New', 'Bellini', 'Cocktail', '>', 'good'], ...","[['New', 'Bellini', 'Cocktail', '>', 'good'], ...",New Bellini Cocktail > good ! Amazing design i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[10, 10, 10, 5, 10, 10, 8, 10, 5, 10, 5, 5, 5,...",8.529412
2,83358,attraction,Wine Bar,Le Baron Rouge,Paris,1 rue Théophile Roussel,48.849623,2.377437,"[{'language': 'en', 'polarity': 5, 'rating': 0...","[[('stop', 'VB'), ('Marche', 'NNP'), ('Aligre'...","[['stop', 'Marche', 'Aligre', 'try', 'wine', '...","[['Marche', 'Aligre', 'glass', 'yummy', 'chees...","[['stop', 'Marche', 'Aligre', 'try', 'glass', ...","[['Marche', 'Aligre', 'wine', 'glass', 'yummy'...","[['stop', 'Marche', 'Aligre', 'try', 'wine', '...",stop by after Marche Aligre to try wine by the...,"[0, 0, 0, 0, 0, 0, 0, 0, 0]","[5, 10, 0, 5, 5, 5, 5, 5, 8]",5.333333
3,83363,attraction,Music Venue,Glazart,Paris,7-15 Avenue de la Porte de la Villette,48.899198,2.386726,"[{'language': 'en', 'polarity': 5, 'rating': 0...","[[('Visit', 'NNP'), ('new', 'JJ'), ('exhibitio...","[['Visit', 'new', 'exhibition', 'Maze', 'Labyr...","[['Visit', 'exhibition', 'Maze', 'Labyrinth', ...","[['Visit', 'exhibition', 'Maze', 'Labyrinth', ...","[['Visit', 'new', 'exhibition', 'Maze', 'Labyr...","[['Visit', 'new', 'exhibition', 'Maze', 'Labyr...","Visit of the new exhibition Maze after ""Labyri...","[0, 0, 0, 0]","[5, 10, 5, 0]",5.000000
4,83364,attraction,Pub,O'Sullivan's by the mill : Backstage by the mill,Paris,92 Boulevard de Clichy,48.883926,2.332180,"[{'language': 'en', 'polarity': 5, 'rating': 0...","[[('Irish', 'JJ'), ('skip', 'NN'), ('que', 'NN...","[['Irish', 'skip', 'que', 'Just', 'say', ""'m"",...","[['skip', 'que', 'Just'], ['Initiation', 'Sals...","[['skip', 'que', 'Just', 'say', ""'m""], ['Initi...","[['Irish', 'skip', 'que', 'Just', 'Irish'], ['...","[['Irish', 'skip', 'que', 'Just', 'say', ""'m"",...","Irish can skip the que. Just say 'Kiss me, I'm...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[5, 5, 10, 5, 10, 8, 5, 5, 10, 5, 10, 10, 5, 10]",7.357143


### Generate Text Vectors

In [13]:
# Prepare documents: combine destination name + review text for richer context
documents = [
    f"Destination: {row['name']}, Address: {row['address']}. Review: {row['clean_reviews']}"
    for _, row in df.iterrows()
]

# Initialize the embedding model
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Generate embeddings in batches
print("Generating embeddings...")
embeddings = list(embedding_model.embed(documents))

Generating embeddings...


### Prepare Payloads

In [ ]:
# Create payloads (metadata for filtering)
payloads = [
    {
        "subCategory": row["subCategory"],
        "name": row["name"],
        "address": row["address"],
        "lat": row["lat"],
        "lng": row["lng"],
        "polarities": row["polarities"],
        "reviews": row["clean_reviews"],
        "avg_polarity": row["avg_polarity"],
    }
    for _, row in df.iterrows()
]

### Create Points

In [15]:
# Create points 
points = [
    models.PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload=payload
    )
    for idx, (embedding, payload) in enumerate(zip(embeddings, payloads))
]

### Insert Points into Collection

Check that points were inserted and vector config is correct.

In [16]:
# Upsert generated points into the collection
client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Indexed {len(points)} points successfully!")

Indexed 477 points successfully!


## Step 11: Verify Collection

In [17]:
collection_info = client.get_collection(collection_name)
print(f"Collection contains {collection_info.points_count} points")
print(f"Vector size: {collection_info.config.params.vectors.size}")
print(f"Distance metric: {collection_info.config.params.vectors.distance}")

Collection contains 477 points
Vector size: 384
Distance metric: Cosine


## Part 3: Similarity Search and Visualization

## Step 12: Perform Vector Search Without Any Payload Filtering


In [18]:
def semantic_search(query, limit=5, filter_category=None):
    """
    Perform semantic search without any payload filtering
    Args:
        query (str): User's natural language query
        limit (int): Number of results to return
    Returns:
        list of ScoredPoint objects with score and payload
    """

    # 1. Convert query to embedding (generator -> next() gives first vector)
    query_embedding = next(embedding_model.embed([query])) 
    
    # 2. Search in Qdrant
    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),  # Pass vector directly
        limit=limit,
        with_payload=True  # Include metadata in results
    ).points

    return results


In [19]:
# Query with a semantic search
# Example: Find peaceful coffee shops with a view
query = "peaceful coffee shops with a view of the Eiffel Tower"
results = semantic_search(query, limit=3)
print(results)


[ScoredPoint(id=417, version=5, score=0.73892295, payload={'subCategory': 'Bar', 'name': "L'Adada Bar", 'address': '15 Rue du Maine', 'lat': 48.84062053, 'lng': 2.32221365, 'polarities': [10], 'reviews': 'Nice atmosphere, calm, nice people, awesome photobooth for only 2 euros, reasonably priced', 'avg_polarity': 10.0}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=40, version=5, score=0.7226324, payload={'subCategory': 'Historic Site', 'name': 'Palais Royal', 'address': 'Place du Palais Royal', 'lat': 48.86357138, 'lng': 2.337019444, 'polarities': [5, 10, 5, 9, 0], 'reviews': "Nice place to start any kind of touristic running session. Beautiful garden, amazing window shopping, fantastic coffee! shop, eat, chill. this is what shopping malls should be like. It’s a lovely area for a quiet afternoon of boutique browsing. When we need a break from shopping, we like to wander through the tranquil gardens or sip a café au lait at one of the refined cafés. One of my favourite 

In [20]:
for result in results:
    print(f"Score: {result.score:.4f}")
    print(f"Destination: {result.payload.get('name')}")
    print(f"Address: {result.payload.get('address')}")
    print(f"Rating: {result.payload.get('avg_polarity')}")
    print(f"Review snippet: {result.payload.get('reviews', '')[:100]}...")
    print("-" * 50)

Score: 0.7389
Destination: L'Adada Bar
Address: 15 Rue du Maine
Rating: 10.0
Review snippet: Nice atmosphere, calm, nice people, awesome photobooth for only 2 euros, reasonably priced...
--------------------------------------------------
Score: 0.7226
Destination: Palais Royal
Address: Place du Palais Royal
Rating: 5.8
Review snippet: Nice place to start any kind of touristic running session. Beautiful garden, amazing window shopping...
--------------------------------------------------
Score: 0.7170
Destination: Quai d'Orsay - Voie sur Berge
Address: Quai d'Orsay
Rating: 5.0
Review snippet: Nice pedestrian area along the river with some great cafes. Perfect place to get out of the sun and ...
--------------------------------------------------


## Step 13: Semantic Search with Payload Filtering

Filtering allows narrowing results based on metadata. Common filter types:
- `MatchValue`: exact match on a string or number.
- `Range`:  numeric/date range (e.g., `gte`, `lte`).
- `MatchAny`: list of possible values.

> **Note:** Since `polarities` is a list, we use `avg_polarity` (scalar) for rating-based filtering.

### Semantic Search with Filters

In [21]:
def semantic_search_with_filters(query, limit=5, min_rating=None, category=None):
    # Vector search with optional filters on avg_polarity and subCategory.
    query_embedding = next(embedding_model.embed([query]))
    # Build filter conditions
    must_conditions = []
    if min_rating is not None:
        must_conditions.append(models.FieldCondition(
                                key="avg_polarity",
                                range=models.Range(gte=min_rating))) # gte: greater than or equal ( >= )
    if category is not None:
        must_conditions.append(models.FieldCondition(
                                key="subCategory",
                                match=models.MatchValue(value=category)))
    search_filter = models.Filter(must=must_conditions) if must_conditions else None
    results = client.query_points(collection_name=collection_name,
                                query=query_embedding.tolist(),
                                limit=limit,                
                                query_filter=search_filter,
                                with_payload=True
                                ).points
    return results
    

In [22]:
# Example 1: Find a highly-rated Bars (avg polarity >= 7)
query = "Good wine and cheese"
results = semantic_search_with_filters(query, limit=3, min_rating=7.0, category="Bar")
print(f"Filtered results for '{query}':")
for result in results:
    print(f"{result.payload['name']} (avg rating: {result.payload['avg_polarity']:.1f}) | Score: {result.score:.4f}")

Filtered results for 'Good wine and cheese':
Le Vin Cœur (avg rating: 8.3) | Score: 0.7468
Café A (avg rating: 10.0) | Score: 0.7061
Le Passy (avg rating: 8.5) | Score: 0.7050


In [24]:
# Example 2: Find several highly-rated lounges (avg polarity >= 6)
queries = [
    "Romantic sunset spots along the Seine",
    "Family friendly hangout",
    "Romantic getaways for couples",
    "Jazzy, cozy, fizzy meeting point near Iron Tower",
]

for q in queries:
    print(f"\n🔍 Query: {q}")
    results = semantic_search_with_filters(q, limit=3, min_rating=6.0, category="Lounge")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r.payload['name']} (score: {r.score:.3f})")


🔍 Query: Romantic sunset spots along the Seine
  1. La Piscine Saint Louis (score: 0.670)
  2. Le Nüba (score: 0.659)
  3. Workshop Paris (score: 0.652)

🔍 Query: Family friendly hangout
  1. Café Barge (score: 0.636)
  2. Tiny Café (score: 0.619)
  3. Workshop Paris (score: 0.617)

🔍 Query: Romantic getaways for couples
  1. La Piscine Saint Louis (score: 0.619)
  2. Workshop Paris (score: 0.609)
  3. Café Barge (score: 0.594)

🔍 Query: Jazzy, cozy, fizzy meeting point near Iron Tower
  1. Tiny Café (score: 0.630)
  2. Café Barge (score: 0.613)
  3. L'Inconnu (score: 0.585)


## Step 14: Visualize the Results

Qdrant provides a built-in visualization tool for exploring our points.
1. Open the Web UI: http://localhost:6333/dashboard
2. Select the collection (left sidebar): **french_tourism**
3. Go to the "Visualize" menu.
4. Run with the desired payload and number of nodes.

```json
{
  "limit": 500,
  "color_by": {
    "payload": "subCategory"
    }
}

## Step 15: Best Practices & Next Steps

***Performance Tips***

- **Batch upload**: For >10k points, use `client.upload_points()` (handles lazy batching, retries, parallelism).
- **Index payload fields**: If you filter frequently (e.g., `subCategory`), create a payload index to speed up queries.
- **Limit search results**: Keep `limit` between 10–100; use `score_threshold` to exclude low-quality matches.

 ***Potential Improvements for This Dataset***

- **Geographic search**: Qdrant supports geo-radius filtering (`geo_radius` on `lat`/`lng`) payload.
- **Hybrid search**: Combine dense (semantic) + sparse (BM25) for better keyword matching on proper nouns.
- **Reranking**: Use a cross-encoder to reorder top-k results for higher precision.

### Study Data Visually

Explore the uploaded data in the Qdrant Web UI at http://localhost:6333/dashboard to study semantic similarity visually.

For example, using the Visualize tab in the `french_tourism`, we can view all answers to the course questions (477 points) and see how they group together by meaning, additionally coloured by the POI address.

To do that, run the following command:

{
  "limit": 500,
  "color_by": {
                 "payload": "address"
                }
}

This 2D representation is the result of dimensionality reduction applied to jina-embeddings.

## Appendix: Experimental / Optional Cells

### Semantic embeddings

In [10]:
from sentence_transformers import SentenceTransformer

# Load pretrained Sentence Transformer models, https://huggingface.co/sentence-transformers/
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    reviews_df["clean_reviews"].tolist(),
    show_progress_bar=True
)

df["embedding"] = list(embeddings)

df["embedding"]

Batches: 100%|██████████| 15/15 [00:01<00:00, 12.51it/s]


0      [0.024902701, -0.029340439, 0.024966799, 0.009...
1      [0.037999164, -0.06289664, 0.036872834, -0.025...
2      [0.04869551, 0.016540756, 0.07132458, -0.03703...
3      [0.006336794, -0.11180983, 0.013949867, 0.0043...
4      [-0.03985235, -0.028476093, -0.040782988, -0.0...
                             ...                        
472    [0.030085843, -0.0033440802, 0.020576792, -0.0...
473    [0.03088291, 0.023995755, -0.056573443, -0.071...
474    [-0.045994245, 0.015373516, 0.008522429, 0.015...
475    [-0.01950341, 0.008807799, 0.052954577, 0.0139...
476    [-0.017785598, -0.0016769087, -0.00018125045, ...
Name: embedding, Length: 477, dtype: object

### Keyword vectors

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20)
X = vectorizer.fit_transform(reviews_df["clean_reviews"])

print(vectorizer.get_feature_names_out())

['and' 'at' 'best' 'for' 'good' 'great' 'in' 'is' 'it' 'nice' 'of' 'on'
 'paris' 'place' 'the' 'this' 'time' 'to' 'with' 'you']


### Sentiment (ranking signal)

In [12]:
from textblob import TextBlob

df["sentiment_score"] = reviews_df["clean_reviews"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

df["sentiment_score"]

0      0.378824
1      0.322546
2      0.033333
3      0.505195
4      0.462890
         ...   
472    0.900000
473    0.500000
474    0.789286
475    0.390000
476   -1.000000
Name: sentiment, Length: 477, dtype: float64

In [ ]:
def get_sentiment_label(score):
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["sentiment_score"].apply(get_sentiment_label)

In [35]:
# Example: Using the Distance Matrix API
import random
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

collection_name = "french_tourism"

# Step 1: scroll to get some points
points, _ = client.scroll(
    collection_name=collection_name,
    limit=50,  # fetch a batch
    with_vectors=True
)

# Step 2: randomly sample 10 points
sampled_points = random.sample(points, min(10, len(points)))

# Step 3: find similar points for each
for p in sampled_points:
    results = client.query_points(
        collection_name=collection_name,
        query=p.vector,
        limit=3
    ).points

    for r in results:
        print(f"Point {p.id} <-> Point {r.id}: score={r.score:.4f}")

Point 5 <-> Point 5: score=1.0000
Point 5 <-> Point 122: score=0.8809
Point 5 <-> Point 391: score=0.8784
Point 28 <-> Point 28: score=1.0000
Point 28 <-> Point 175: score=0.8544
Point 28 <-> Point 41: score=0.8266
Point 14 <-> Point 14: score=1.0000
Point 14 <-> Point 109: score=0.8582
Point 14 <-> Point 87: score=0.8581
Point 45 <-> Point 45: score=1.0000
Point 45 <-> Point 305: score=0.8930
Point 45 <-> Point 306: score=0.8434
Point 3 <-> Point 3: score=1.0000
Point 3 <-> Point 255: score=0.8672
Point 3 <-> Point 351: score=0.8628
Point 8 <-> Point 8: score=1.0000
Point 8 <-> Point 113: score=0.8596
Point 8 <-> Point 199: score=0.8556
Point 12 <-> Point 12: score=1.0000
Point 12 <-> Point 113: score=0.9199
Point 12 <-> Point 249: score=0.8869
Point 4 <-> Point 4: score=1.0000
Point 4 <-> Point 113: score=0.8366
Point 4 <-> Point 87: score=0.8283
Point 43 <-> Point 43: score=1.0000
Point 43 <-> Point 381: score=0.8600
Point 43 <-> Point 342: score=0.8474
Point 1 <-> Point 1: score=1.

In [ ]:
### 🎨 Distace Matrix API

# Example: Using the Distance Matrix API
import random
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

collection_name = "french_tourism"

# Step 1: scroll to get some points
points, _ = client.scroll(
    collection_name=collection_name,
    limit=50,  # fetch a batch
    with_vectors=True
)

# Step 2: randomly sample 10 points
sampled_points = random.sample(points, min(10, len(points)))

# Step 3: find similar points for each
for p in sampled_points:
    results = client.query_points(
        collection_name=collection_name,
        query=p.vector,
        limit=3
    ).points

    for r in results:
        print(f"Point {p.id} <-> Point {r.id}: score={r.score:.4f}")


## ---------Important to Know---------

## Add more points for existing collections

In [ ]:
# Create New Points
"""
new_points = [
    models.PointStruct(
        id=new_id,
        vector=new_embedding,
        payload=new_payload
    )
]
client.upsert(collection_name="french_tourism", points=new_points)
"""

## Delete Collections


In [ ]:
# ***Delete All Data***
# Delete a collection when no longer needed
# > ⚠️  **Remember: this permanently deletes all data**
# client.delete_collection(collection_name="french_tourism")

# ***Or delete specific points***

""" client.delete(
    collection_name="french_tourism",
    points_selector=models.PointIdsList(
        points=[0, 1, 2]  # Delete points with these IDs
    )
) """
